# Dashboard Qualité de Service — SNCB / Infrabel

**Auteur** : Tahar Guenfoud    
**Source** : [Open Data Infrabel](https://opendata.infrabel.be)

---

## Objectif
Construire un dashboard interactif pour analyser la ponctualité et la fiabilité du réseau ferroviaire belge.

## Plan du notebook
| Étape | Description |
|---|---|
| **1. Extract** | Téléchargement des 5 sources Open Data Infrabel |
| **2. Transform** | Nettoyage, typage, colonnes calculées |
| **3. EDA** | Analyse exploratoire — tendances, anomalies |
| **4. KPIs** | Calcul des indicateurs métier (Ponctualité, Reliability, Minutes perdues) |
| **5. Visualisation** | Graphiques pour le dashboard |

---
## 0. Imports & Configuration

In [63]:
import requests
import pandas as pd
import os
import time

# Dossier de sauvegarde
RAW_DIR   = "data/raw"
CLEAN_DIR = "data/clean"

print("✅ Imports OK")

✅ Imports OK


---
# ÉTAPE 1 — Extract
Téléchargement des 5 datasets depuis l'API Open Data Infrabel.

In [64]:
BASE = (
    "https://opendata.infrabel.be/api/explore/v2.1/catalog/datasets"
    "/{}/exports/csv?lang=fr&timezone=Europe%2FBrussels&use_labels=true&delimiter=%3B"
)

DATASETS = {
    "ponctualite_par_gare"    : "maandelijkse-stiptheid-per-stopplaats",
    "causes_retards"          : "oorzaken-vertraging-per-maand",
    "ponctualite_par_moment"  : "nationale-stiptheid-per-moment-en-per-maand",
    "trains_supprimes"        : "afgeschafte-treinen-per-maand-vanaf-2020",
    "kpi_contrat_performance" : "indicatoren-performantie-contract",
}

dfs = {}
for name, dataset_id in DATASETS.items():
    dfs[name] = pd.read_csv(BASE.format(dataset_id), sep=";")
    print(f"✅ {name:<35} {dfs[name].shape[0]:>6,} lignes × {dfs[name].shape[1]} cols")

✅ ponctualite_par_gare                27,902 lignes × 13 cols
✅ causes_retards                         430 lignes × 14 cols
✅ ponctualite_par_moment                 488 lignes × 9 cols
✅ trains_supprimes                        74 lignes × 7 cols
✅ kpi_contrat_performance                177 lignes × 13 cols


---
# ÉTAPE 2 — Transform
Nettoyage et préparation de chaque dataset.

### 2.1 — Ponctualité par Gare

In [ ]:
# Explorer les colonnes brutes
df = dfs["ponctualite_par_gare"]
df.head(10)

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df["Date"].unique()

In [ ]:
df_gare = dfs["ponctualite_par_gare"].copy()

# Renommer (13 colonnes dans l'ordre exact)
df_gare.columns = [
    "date",
    "nom_gare_fr",
    "nom_gare_nl",
    "nom_gare_de",
    "id_gare",
    "classification_fr",
    "classification_nl",
    "classification_de",
    "ponctualite_pct",
    "nb_trains",
    "nb_trains_ponctuels",
    "geo_point",
    "geo_shape"
]

# Parser la date
df_gare["date"] = pd.to_datetime(df_gare["date"], format="%Y-%m")

# Vérification
print(f"Shape : {df_gare.shape}")
print(f"Valeurs manquantes :\n{df_gare.isnull().sum()}")
df_gare.head(3)

In [ ]:
# Voir quelques exemples côte à côte
df_gare[["nom_gare_fr", "nom_gare_nl", "nom_gare_de"]].drop_duplicates().head(20)

In [ ]:
# Combien de fois les 3 colonnes sont identiques ?
identiques = (df_gare["nom_gare_fr"] == df_gare["nom_gare_nl"]).sum()
print(f"FR = NL : {identiques} fois sur {len(df_gare)}")

identiques2 = (df_gare["nom_gare_fr"] == df_gare["nom_gare_de"]).sum()
print(f"FR = DE : {identiques2} fois sur {len(df_gare)}")

In [ ]:
# Supprimer les colonnes inutiles
df_gare = df_gare.drop(columns=["nom_gare_nl", "nom_gare_de",
                                 "classification_nl", "classification_de",
                                 "geo_shape"])

print(f"Colonnes restantes : {df_gare.columns.tolist()}")
print(f"Shape : {df_gare.shape}")

In [ ]:
display(df_gare.head())
df_gare.info()

### 2.2 — Causes des Retards

### Colonnes clés — Causes des Retards

| Colonne | Description |
|---|---|
| `responsable` | Qui a causé le retard : Infrabel / SNCB / Tiers / Robustesse systémique / Autres |
| `nb_trains_en_retard` | Nombre de trains en retard imputés à ce responsable ce mois |
| `nb_trains_total` | Nombre total de trains observés ce mois |
| `perte_ponctualite` | Minutes de retard totales causées par ce responsable ce mois |
| `proportion_pct` | Part (%) de ce responsable dans les retards du mois |
| `nb_trains_en_retard_ytd` | Cumul des trains en retard depuis le 1er janvier (YTD) |
| `nb_trains_total_ytd` | Cumul total des trains depuis le 1er janvier (YTD) |
| `perte_ponctualite_ytd` | Cumul des minutes de retard depuis le 1er janvier (YTD) |
| `proportion_ytd_pct` | Part (%) cumulée depuis le 1er janvier (YTD) |

> **YTD (Year-To-Date)** : cumul depuis le 1er janvier de l'année en cours.
> Permet de suivre la tendance annuelle indépendamment des variations mensuelles.

In [ ]:
# Explorer les colonnes brutes
df = dfs["causes_retards"]
display(df.head(3))
df.info()

In [ ]:
df_causes = dfs["causes_retards"].copy()

print(f"Shape : {df_causes.shape}")
print(f"Valeurs manquantes :\n{df_causes.isnull().sum()}")
df_causes.head(3)

In [ ]:
# Valeurs uniques dans chaque colonne
print("FR :", df["Responsable"].unique())
print("NL :", df["Responsable NL"].unique())
print("EN :", df["Responsable EN"].unique())

In [ ]:
# Vérifier si FR et NL sont toujours différents
identiques = (df["Responsable"] == df["Responsable NL"]).sum()
print(f"FR = NL : {identiques} fois sur {len(df)}")

identiques2 = (df["Responsable"] == df["Responsable EN"]).sum()
print(f"FR = EN : {identiques2} fois sur {len(df)}")

In [ ]:
df_causes.columns = [
    "annee", "date", "mois",
    "responsable_nl", "responsable", "responsable_en",
    "nb_trains_en_retard", "nb_trains_total",
    "perte_ponctualite", "proportion_pct",
    "nb_trains_en_retard_ytd", "nb_trains_total_ytd",
    "perte_ponctualite_ytd", "proportion_ytd_pct"
]

# Parser la date
df_causes["date"] = pd.to_datetime(df_causes["date"], format="%Y-%m")

# Supprimer les colonnes redondantes
df_causes = df_causes.drop(columns=["annee", "mois", "responsable_nl", "responsable_en"])

print(f"Shape : {df_causes.shape}")
print(f"\nResponsables : {df_causes['responsable'].unique()}")
df_causes.head()

In [ ]:
df_causes.info()

### 2.3 — Ponctualité par Moment

In [ ]:
df = dfs["ponctualite_par_moment"]
display(df.head())
df.info()

In [ ]:
print(df["Instant"].unique())
print(df["Instant.1"].unique())
print(df["Instant.2"].unique())

In [ ]:
df_moment = dfs["ponctualite_par_moment"].copy()

df_moment.columns = [
    "date",
    "periode_nl",
    "periode",
    "periode_en",
    "ponctualite_pct",
    "nb_trains",
    "nb_trains_ponctuels",
    "nb_minutes_retard",
    "annee"
]

# Parser la date
df_moment["date"] = pd.to_datetime(df_moment["date"], format="%Y-%m")

# Supprimer les colonnes redondantes
df_moment = df_moment.drop(columns=["periode_nl", "periode_en", "annee"])

print(f"Shape : {df_moment.shape}")
print(f"\nPériodes disponibles : {df_moment['periode'].unique()}")
df_moment.head()

In [ ]:
df_moment.info()

### 2.4 — Trains Supprimés

In [ ]:
df = dfs["trains_supprimes"]
display(df.head(3))
df.info()

In [ ]:
df_suppression = dfs["trains_supprimes"].copy()

df_suppression.columns = [
    "date",
    "nb_trains_supprimes_total",
    "nb_trains_supprimes_partiel",
    "nb_trains_supprimes_entier",
    "nb_trains",
    "pct_trains_supprimes",
    "annee"
]

# Parser la date
df_suppression["date"] = pd.to_datetime(df_suppression["date"], format="%Y-%m")

# Supprimer colonne redondante
df_suppression = df_suppression.drop(columns=["annee"])

print(f"Shape : {df_suppression.shape}")
df_suppression.head()

In [ ]:
df_suppression.info()

### 2.5 — KPIs Contrat de Performance

In [ ]:
df = dfs["kpi_contrat_performance"]
display(df.head(3))
df.info()

In [ ]:
df.isna().sum()

In [ ]:
# Quelles catégories existent ?
print("Catégories :")
print(df["Catégorie"].value_counts())

print("\nTypes :")
print(df["Type"].value_counts())

print("\nSous-catégories :")
print(df["Sous-catégorie"].value_counts())

*
J'ai supprimé les colonnes avec plus de 50% de valeurs manquantes et sans valeur ajoutée pour le dashboard. J'ai conservé objectif et valeur_reelle malgré leurs valeurs manquantes car ce sont les deux colonnes centrales pour comparer performance réelle vs objectif contractuel 


In [ ]:
df_kpi = dfs["kpi_contrat_performance"].copy()

# Renommer
df_kpi.columns = [
    "annee", "id_indicateur", "type_indicateur",
    "categorie_nl", "categorie",
    "sous_categorie_nl", "sous_categorie",
    "objectif", "valeur_reelle", "bonus",
    "remediation", "seuil_superieur", "unite"
]

# Garder seulement Ponctualité + Fiabilité
df_kpi = df_kpi[df_kpi["categorie"].isin(["Ponctualité", "Fiabilité & Vitesse Commerciale"])]

# Supprimer colonnes inutiles
df_kpi = df_kpi.drop(columns=[
    "categorie_nl", "sous_categorie_nl",
    "seuil_superieur",
    "bonus", "remediation"
])

print(f"Shape : {df_kpi.shape}")
print(f"\nIndicateurs restants :")
print(df_kpi["sous_categorie"].unique())
df_kpi.head()

In [ ]:
df_kpi.info()

In [ ]:
# !git add .
# !git commit -m "feat: étape 2 — Transform des 5 datasets (nettoyage, renommage, parsing dates)"
# !git push

---
# 📊 ÉTAPE 3 — EDA (Exploratory Data Analysis)

> **Objectif :** Explorer les données pour découvrir des insights avant de construire le dashboard.

## Plan EDA

| Section | Question |
|---|---|
| **3.1** Tendance temporelle | La ponctualité s'améliore ou empire sur les dernières années ? |
| **3.2** Top 10 gares problématiques | Quelles gares cumulent le plus de retards ? |
| **3.3** Analyse par moment | Matin vs Soir vs Heures creuses vs Weekend ? |
| **3.4** Causes des retards | Quelle part pour Infrabel, SNCB, Tiers ? |
| **3.5** Trains supprimés | Comment évolue la fiabilité du réseau ? |

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio                                                       
pio.renderers.default = "notebook"



In [ ]:
df_gare["date"].unique()

### 3.1 — Tendance Temporelle
> La ponctualité nationale s'améliore-t-elle au fil des années ?

In [ ]:
# Agréger par mois (moyenne nationale)
df_tendance = df_gare.groupby("date")["ponctualite_pct"].mean().reset_index()
display(df_tendance.head())
# Graphique
fig = px.line(
    df_tendance,
    x="date",
    y="ponctualite_pct",
    title="📈 Tendance de la Ponctualité Nationale (2022-2025)",
    labels={"date": "Mois", "ponctualite_pct": "Ponctualité (%)"},
)

# Ligne seuil objectif contractuel (90%)
fig.add_hline(
    y=90,
    line_dash="dash",
    line_color="red",
    annotation_text="Objectif 90%"
)

fig.show()

In [ ]:
# Zoomer sur la période août-novembre 2023
masque = (df_causes["date"] >= "2023-08-01") & (df_causes["date"] <= "2023-11-30")
df_crise = df_causes[masque]

# Qui est responsable ?
print(df_crise.groupby("responsable")["perte_ponctualite"].sum().sort_values(ascending=False))

In [ ]:
# Moyenne annuelle de ponctualité
df_annuel = df_gare.groupby(df_gare["date"].dt.year)["ponctualite_pct"].mean().reset_index()
df_annuel.columns = ["annee", "ponctualite_moyenne"]
df_annuel["ponctualite_moyenne"] = df_annuel["ponctualite_moyenne"].round(2)
print(df_annuel)

In [ ]:
# Filtrer depuis janvier 2024
df_2024_plus = df_tendance[df_tendance["date"] >= "2024-01-01"].copy()

# Calculer la pente (régression linéaire simple)
from numpy.polynomial import polynomial as P
import numpy as np

x = np.arange(len(df_2024_plus))
y = df_2024_plus["ponctualite_pct"].values
coefs = P.polyfit(x, y, 1)

print(f"Pente mensuelle : +{coefs[1]:.3f}% par mois")
print(f"Soit +{coefs[1]*12:.2f}% par an")


In [ ]:
!git add .
!git commit -m "feat: étape 3 — Tendance Temporelle"
!git push

### 3.2 Top 10 gares problématiques	
> Quelles gares cumulent le plus de retards ?

In [ ]:
df_gare.head()

In [ ]:
# Calculer le nombre de trains en retard par ligne
df_gare['nb_trains_retard'] = df_gare['nb_trains'] - df_gare['nb_trains_ponctuels']

# Grouper par gare pour avoir le cumul total de trains en retard
top_10_volume = df_gare.groupby("nom_gare_fr")['nb_trains_retard'].sum().sort_values(ascending=False).head(10).reset_index()
display(top_10_volume)

In [ ]:
# 1. Calculer le volume de retards et la ponctualité moyenne par gare
df_analyse_gares = df_gare.groupby("nom_gare_fr").agg({
    'nb_trains_retard': 'sum',
    'ponctualite_pct': 'mean'
}).reset_index()

# 2. Prendre le Top 10 par volume de retards
top_10_final = df_analyse_gares.sort_values(by="nb_trains_retard", ascending=False).head(10)

# 3. Arrondir pour la lisibilité
top_10_final['ponctualite_pct'] = top_10_final['ponctualite_pct'].round(2)

In [ ]:
import plotly.express as px

fig = px.bar(
    top_10_final,
    x="nb_trains_retard",
    y="nom_gare_fr",
    orientation='h',
    color="ponctualite_pct", # La couleur montre la performance
    color_continuous_scale="RdYlGn", # Rouge (mauvais) -> Jaune -> Vert (bon)
    title="🚆 Top 10 des Gares par Volume de Retards (Impact Passager)",
    labels={
        "nb_trains_retard": "Nombre Total de Trains en Retard",
        "nom_gare_fr": "Gare",
        "ponctualite_pct": "Ponctualité Moyenne (%)"
    },
    text="nb_trains_retard" # Affiche le chiffre exact au bout de la barre
)

# Amélioration du design
fig.update_traces(textposition='outside')
fig.update_layout(
    yaxis={'categoryorder':'total ascending'}, # Met la pire gare (Brussel-Zuid) en haut
    plot_bgcolor='white',
    xaxis_title="Volume de retards (Cumulé)",
    height=600
)

fig.show()

In [ ]:
# !git add .
# !git commit -m "feat: étape 3 — 3.2 Top 10 gares problématiques"
# !git push

### 3.3 Analyse par moment	
> Matin vs Soir vs Heures creuses vs Weekend ?

In [ ]:
display(df_moment.head())
df_moment.info()

In [ ]:
df_moment["periode"].unique()

In [ ]:
display(df_analyse_gares.head())
df_analyse_gares.info()

In [ ]:
df_moment["df_moment_retard"] = df_moment["nb_trains"] - df_moment["nb_trains_ponctuels"]
periode = df_moment.groupby("periode")['df_moment_retard'].sum().sort_values(ascending=False).head(10).reset_index()
display(periode)

In [ ]:
df_moment.groupby("periode")["ponctualite_pct"].mean().sort_values().round(2)

In [ ]:
df_par_periode = df_moment.groupby("periode").agg(
    nb_trains_retard=("df_moment_retard", "sum"),
    ponctualite_pct=("ponctualite_pct", "mean")
).reset_index().round(2)

display(df_par_periode.sort_values("ponctualite_pct"))

In [ ]:
fig = px.bar(
    df_par_periode.sort_values("ponctualite_pct"),
    x="periode",
    y="ponctualite_pct",
    color="ponctualite_pct",
    color_continuous_scale="RdYlGn",
    text="ponctualite_pct",
    title="⏰ Ponctualité par Période (Matin / Soir / Creuses / Weekend)",
    labels={"ponctualite_pct": "Ponctualité (%)", "periode": "Période"}
)
fig.add_hline(y=90, line_dash="dash", line_color="red", annotation_text="Objectif 90%")
fig.show()

In [ ]:
!git add .
!git commit -m "feat: étape 3 — 3.3 Analyse par moment"
!git push

### 3.4 Causes des retards	
> Quelle part pour Infrabel, SNCB, Tiers ?

In [ ]:
display(df_causes.head())
df_causes.info()

In [ ]:
# masque = (df_causes["date"] >= "2023-08-01") & (df_causes["date"] <= "2023-11-30")
# df_crise = df_causes[masque]

# Qui est responsable ?
print(df_causes.groupby("responsable")["perte_ponctualite"].sum().sort_values(ascending=False))

In [ ]:
fig = px.pie(
    df_causes.groupby("responsable")["perte_ponctualite"].sum().reset_index(),
    values="perte_ponctualite",
    names="responsable",
    title="🔍 Répartition des Responsables de Retards",
    color_discrete_sequence=px.colors.sequential.RdBu
)
fig.show()

In [ ]:
!git add .
!git commit -m "feat: étape 3 — 3.4 Causes des retards"
!git push

### 3.5 Trains supprimés	
> Comment évolue la fiabilité du réseau ?

In [ ]:
display(df_suppression.head())
df_suppression.info()

In [ ]:
# 3.5 — Fiabilité du réseau (trains non supprimés)
import numpy as np

# Fiabilité = 100% - % trains supprimés
df_suppression["fiabilite_pct"] = (100 - df_suppression["pct_trains_supprimes"]).round(2)

# Moyenne mobile 3 mois pour lisser
df_suppression["fiabilite_lissee"] = df_suppression["fiabilite_pct"].rolling(window=3, center=True).mean().round(2)

# Stats rapides
moy = df_suppression["fiabilite_pct"].mean().round(2)
pire = df_suppression["fiabilite_pct"].min()
meilleur = df_suppression["fiabilite_pct"].max()
print(f"Fiabilité moyenne  : {moy}%")
print(f"Pire mois          : {pire}%")
print(f"Meilleur mois      : {meilleur}%")

# Graphique amélioré
fig = go.Figure()

# Zone colorée en arrière-plan (fill sous la courbe lissée)
fig.add_trace(go.Scatter(
    x=df_suppression["date"],
    y=df_suppression["fiabilite_lissee"],
    fill="tozeroy",
    fillcolor="rgba(99, 155, 255, 0.10)",
    line=dict(color="rgba(0,0,0,0)"),
    showlegend=False,
    hoverinfo="skip"
))

# Courbe brute (fine, transparente) — le "bruit"
fig.add_trace(go.Scatter(
    x=df_suppression["date"],
    y=df_suppression["fiabilite_pct"],
    mode="lines",
    line=dict(color="rgba(99, 155, 255, 0.30)", width=1),
    name="Mensuel brut"
))

# Courbe lissée (épaisse) — la tendance claire
fig.add_trace(go.Scatter(
    x=df_suppression["date"],
    y=df_suppression["fiabilite_lissee"],
    mode="lines",
    line=dict(color="#3b82f6", width=3),
    name="Moyenne mobile 3 mois"
))

# Ligne objectif
fig.add_hline(
    y=99, line_dash="dash", line_color="red", line_width=1.5,
    annotation_text="Objectif 99%", annotation_position="top right"
)

# Ligne moyenne globale
fig.add_hline(
    y=moy, line_dash="dot", line_color="orange", line_width=1.5,
    annotation_text=f"Moyenne {moy}%", annotation_position="bottom right"
)

fig.update_layout(
    title=dict(text="🚫 Évolution de la Fiabilité — % Trains Non Annulés", font_size=16),
    xaxis_title="Mois",
    yaxis_title="Fiabilité (%)",
    yaxis=dict(range=[93, 100]),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    plot_bgcolor="white",
    hovermode="x unified",
    height=450
)
fig.update_xaxes(showgrid=True, gridcolor="#f0f0f0")
fig.update_yaxes(showgrid=True, gridcolor="#f0f0f0")

fig.show()


---
# ÉTAPE 4 — KPIs Métier
> 🔜 À compléter ensemble après l'EDA

---
# ÉTAPE 5 — Visualisation
> 🔜 À compléter ensemble après les KPIs